In [1]:
# Cell 1: Import and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('../data/processed/merged_clean.csv', index_col=0, parse_dates=True)

features = ['oil_lag1', 'oil_lag2', 'oil_lag3',
            'gbp_lag1', 'gbp_lag2',
            'wages_lag1', 'wages_lag3',
            'unemployment']

X = df[features]
y = df['cpi']

print("Data loaded :)")
print(f"Latest observation: {df.index[-1].date()}")
print(f"\nLatest values:")
print(df[['cpi', 'oil_price', 'gbp_usd', 'wages', 'unemployment']].tail(3))

Data loaded :)
Latest observation: 2026-05-01

Latest values:
            cpi  oil_price   gbp_usd     wages  unemployment
date                                                        
2026-03-01  3.3     70.887  1.357921  3.112165           5.0
2026-04-01  2.8     70.887  1.357921  3.112165           5.0
2026-05-01  2.8     70.887  1.357921  3.112165           5.0


In [2]:
# Cell 2: Train rolling window model on most recent 60 months to capture recent inflation dynamics

window_size = 60

# Training window: most recent 60 months
X_train_final = X.iloc[-window_size:]
y_train_final = y.iloc[-window_size:]

# Fit model
final_model = LinearRegression()
final_model.fit(X_train_final, y_train_final)

print(f"Model trained on: {X_train_final.index[0].date()} to {X_train_final.index[-1].date()}")
print(f"Training observations: {len(X_train_final)}")

Model trained on: 2021-06-01 to 2026-05-01
Training observations: 60


In [3]:
# Cell 3: Construct feature vector for next month nowcast

# Automatically identify latest available month
latest = df.iloc[-1]   
prev1 = df.iloc[-2]
prev2 = df.iloc[-3]
prev3 = df.iloc[-4]

forecast_month = latest.name + pd.DateOffset(months=1)

features = {
    'oil_lag1':     latest['oil_price'],
    'oil_lag2':     prev1['oil_price'],
    'oil_lag3':     prev2['oil_price'],
    'gbp_lag1':     latest['gbp_usd'],
    'gbp_lag2':     prev1['gbp_usd'],
    'wages_lag1':   latest['wages'],
    'wages_lag3':   prev2['wages'],
    'unemployment': latest['unemployment']
}

print(f"\nFeature values for {forecast_month.strftime('%B %Y')} nowcast:")
for k, v in features.items():
    print(f"  {k}: {v:.3f}")


Feature values for June 2026 nowcast:
  oil_lag1: 70.887
  oil_lag2: 70.887
  oil_lag3: 70.887
  gbp_lag1: 1.358
  gbp_lag2: 1.358
  wages_lag1: 3.112
  wages_lag3: 3.112
  unemployment: 5.000


In [4]:
# Cell 4: Generate next month CPI nowcast

# Convert features into dataframe for model prediction
X_forecast = pd.DataFrame([features])

# Make prediction
forecast_prediction = final_model.predict(X_forecast)[0]

# Calculate uncertainty interval using MAE from rolling window model
# Approximate 95% confidence interval
mae_rolling = 2.22  # from notebook 04
margin = 1.96 * mae_rolling

lower = forecast_prediction - margin
upper = forecast_prediction + margin

forecast_name = forecast_month.strftime('%B %Y')

print("=" * 45)
print(f"   UK CPI NOWCAST: {forecast_name.upper()}")
print("=" * 45)
print(f"   Point estimate:    {forecast_prediction:.2f}%")
print(f"   95% interval:      [{lower:.2f}%, {upper:.2f}%]")
print(f"   Previous CPI:      {df['cpi'].iloc[-1]:.2f}% ({df.index[-1].strftime('%B %Y')})")
print(f"   BoE target:        2.00%")
print(f"   Gap from target:   {forecast_prediction - 2.0:+.2f}pp")
print("=" * 45)

   UK CPI NOWCAST: JUNE 2026
   Point estimate:    3.12%
   95% interval:      [-1.23%, 7.48%]
   Previous CPI:      2.80% (May 2026)
   BoE target:        2.00%
   Gap from target:   +1.12pp
